# Boulder County & Colorado SoS election results ? methodology & worked examples

**Schema version:** 1.0.0 ? **Last rewrite:** 2026-05-23 ? **Maintainer:** Brian Keegan

---

## What this notebook is

This notebook is the **methodology document** for the harmonized dataset in `data/processed/`.
It does *not* re-implement the cleaning ? that lives in [`scripts/parsers/`](scripts/parsers/) and runs through `python -m scripts.pipeline`.
What this notebook does:

1. States the question the data is built to answer and the scope of the project.
2. Documents how raw files become harmonized rows (with pointers to source code).
3. Demonstrates the harmonized data through worked examples a future analyst can lift verbatim.
4. Records reconciliation checks against upstream-published totals ? the *only* way we know the parser is doing the right thing.

All paths are parameterized below; you should be able to run this top-to-bottom on any checkout.

## How to use

```bash
uv sync                                                  # install deps
uv run python -m scripts.pipeline --skip-fetch --no-pdf  # produce data/processed/
uv run jupyter lab Cleaning.ipynb                        # open this notebook
```

## Compliance: public-interest data-journalism notebook standards

- [x] Parameterized paths (no hardcoded `/Users/...`)
- [x] Reproducibility statement (this section)
- [x] Source citations with retrieval timestamps (every row carries `source_url` + `retrieved_at`)
- [x] Methodology documented inline + in `docs/data-dictionary.md`
- [x] Sanity checks against published totals (Section 5 below)
- [x] No mutating `inplace=True`; no chained assignment
- [x] No embedded credentials or PII

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(os.environ.get('ELECTION_REPO_ROOT', '.')).resolve()
PROCESSED = REPO_ROOT / 'data' / 'processed'
LOOKUPS = REPO_ROOT / 'data' / 'lookups'
AUDIT = REPO_ROOT / 'data' / 'audit'

assert PROCESSED.exists(), f'Run pipeline first; expected {PROCESSED}'

df = pd.read_csv(
    PROCESSED / 'all-elections-tidy.csv',
    dtype={'precinct_id': 'string', 'precinct_name': 'string'},
    low_memory=False,
)

print(f'rows: {len(df):,}')
print(f'years: {sorted(df["election_year"].dropna().unique().tolist())}')
print(f'sources: {df["data_source"].value_counts().to_dict()}')

## 2. Sources and provenance

The URL registry lives in [`scripts/sources.py`](scripts/sources.py); every row in the harmonized dataset carries `source_url` and `retrieved_at`.

In [ ]:
provenance = (
    df.groupby(['election_year', 'election_type', 'data_source'])
      .agg(source_file=('source_file', 'first'),
           source_url=('source_url', 'first'),
           retrieved_at=('retrieved_at', 'first'),
           rows=('votes', 'size'))
      .reset_index()
)
provenance.head(10)

## 3. Methodology: how raw files become harmonized rows

Each year/source pair is routed to a parser by [`scripts/clean.py`](scripts/clean.py):

| Years | Source | Parser | Strategy |
|---|---|---|---|
| 2005, 2007, 2009 | Boulder | [`parsers/boco_pdf.py`](scripts/parsers/boco_pdf.py) | pdfplumber text-layer extraction (pure Python, no JVM). Contest titles come from the page text; candidate names are reconstructed by clustering rotated `upright=False` words by x-coordinate and reversing their characters. |
| 2008, 2010, 2011, 2012 | Boulder | [`parsers/boco_panel.py`](scripts/parsers/boco_panel.py) | Walks the sheet detecting `Precinct` header rows; melts each panel wide to long. |
| 2013-2024 | Boulder | [`parsers/boco_tidy.py`](scripts/parsers/boco_tidy.py) | Already long-form. Normalizes column-name variants. |
| 2004-2020 | SOS | [`parsers/sos.py`](scripts/parsers/sos.py) | Filters to Boulder rows. Two generations: 2004-2010 has a single `Votes` column; 2012-2020 splits `Candidate Votes` / `Yes Votes` / `No Votes`. |

Every parser returns the canonical schema in [`scripts/schema.py`](scripts/schema.py).

Why this layout (vs. the old single-notebook approach):

- The original `Cleaning.ipynb` had ~74 code cells with execution gaps and hardcoded row indices.
- Splitting into parsers means each year's logic is testable in isolation; adding a new year is one module + one entry in `sources.py`.
- The notebook becomes the *narrative*, not the production code.

## 4. Schema overview

Full reference: [`docs/data-dictionary.md`](docs/data-dictionary.md).

In [ ]:
df.head(3).T

In [ ]:
df.isna().mean().round(3).sort_values(ascending=False)

## 5. Reconciliation checks

These checks should hold within tolerance. If they fail, the parser regressed.

### 5.1 2020 Presidential ? Boulder County

Sum vote totals by candidate. Biden should win Boulder by a wide margin (Boulder is heavily Democratic). Cross-check against the certified figures linked from the README before publishing analyses.

In [ ]:
pres20 = (
    df[(df['election_year'] == 2020)
       & (df['data_source'] == 'boulder_county')
       & df['contest'].str.contains('Presidential', case=False, na=False)]
    .groupby('candidate_or_option')['votes'].sum()
    .sort_values(ascending=False)
)
pres20.head(10)

### 5.2 Boulder vs. SOS reconciliation (2020 US Senator)

Both Boulder County and the SOS reported the same race. They should match within a tiny tolerance ? any larger gap means one parser is wrong.

In [ ]:
sen20 = (
    df[(df['election_year'] == 2020)
       & df['contest'].str.contains('United States Senator', case=False, na=False)]
    .groupby(['data_source', 'candidate_or_option'])['votes']
    .sum().unstack(0)
)
sen20['delta'] = (sen20.get('boulder_county', 0) - sen20.get('secretary_of_state', 0))
sen20.sort_values('boulder_county', ascending=False, na_position='last').head(10)

### 5.3 Coverage by year and source

In [ ]:
(
    df.groupby(['election_year', 'data_source'])
      .agg(rows=('votes', 'size'),
           contests=('contest', 'nunique'),
           precincts=('precinct_id', 'nunique'))
)

## 6. Worked examples

See [`docs/filter-pivot-recipes.md`](docs/filter-pivot-recipes.md) for the full set of pandas / tidyverse / Excel recipes.

### 6.1 City of Boulder subset

In [ ]:
cob_ids = set(json.loads((LOOKUPS / 'city-of-boulder-precincts-2023.json').read_text()))
print(f'City of Boulder precincts: {len(cob_ids)}')
cob = df[df['precinct_id'].isin(cob_ids)]
print(f'City of Boulder rows across all years: {len(cob):,}')

### 6.2 Ranked-choice mayoral race (2023)

In [ ]:
mayor = df[
    (df['election_year'] == 2023)
    & (df['contest_type'] == 'ranked_choice')
    & df['contest'].str.contains('Mayor', case=False, na=False)
].copy()
mayor[['candidate', 'round_label']] = mayor['candidate_or_option'].str.split(r'\s*\|\s*', expand=True)
mayor.pivot_table(index='candidate', columns='round_label',
                  values='votes', aggfunc='sum', fill_value=0)

### 6.3 Ballot-measure pass rates by year

In [ ]:
measures = (
    df[(df['contest_type'] == 'measure') & (df['election_year'] >= 2014)]
    .groupby(['election_year', 'contest', 'candidate_or_option'])['votes'].sum()
    .unstack('candidate_or_option').fillna(0)
)
measures['passed'] = measures['Yes'] > measures['No']
measures.groupby(level='election_year')['passed'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'passed', 'count': 'total', 'mean': 'pass_rate'}
)

## 7. Known limitations

See `README.md` ? *Known limitations* and `docs/data-dictionary.md` ? 4.2 / 4.4. Highlights:

- **2008 / 2010 Boulder** ? merged-cell panel headers misalign with data columns; major contests only.
- **2005 / 2007 / 2009 PDFs** ? contest titles not extracted from surrounding narrative; rows arrive with placeholder contest names.
- **2024 Boulder** ? `Amended Statement of Votes` upstream file is only 66 rows of local races; verify before publishing.

## 8. How to add a new year

1. Add the entry to `BOULDER_COUNTY` or `SECRETARY_OF_STATE` in [`scripts/sources.py`](scripts/sources.py).
2. Add the certification date to `ELECTION_DATES`.
3. Pick a parser route in [`scripts/clean.py`](scripts/clean.py).
4. Run `uv run python -m scripts.pipeline --year <year>`; spot-check the file in `data/processed/`.
5. Refresh `data/audit/{stem}.md`; copy the per-year row into the table in `docs/data-dictionary.md` ? 5.

The annual GitHub Action does steps 4-5 automatically for the prior calendar year.